# Tanzania Financial News Scraper
### DS & AI Year I: Data Collection Notebook

This notebook collects financial news headlines from major Tanzanian news outlets
for the period **2022 – 2024**. The collected dataset will be used for:

- Sentiment analysis of financial news
- Correlation analysis with TZS/USD exchange rates and inflation data

**Target outlets:**
- Daily News Tanzania (`dailynews.co.tz`)
- The Citizen Tanzania (`thecitizen.co.tz`)

**Authors:** *(add your names)*  
**Date:** *(add date)*


---
## Section 1: Site Configuration

This is the **only section you need to edit** when adding a new website.

Each site is defined as a Python dictionary with five fields.
The scraping functions in Section 4 read from these dictionaries automatically
you never need to touch the functions.

### How to find the selectors for a new site
1. Open the site in Chrome/Firefox
2. Right-click an article headline → **Inspect**
3. Look at the HTML structure around the headline
4. Identify: what tag/class wraps the list of articles? What tag/class is the headline link inside?

### How to find the paginated URL pattern
Navigate to page 2 of the category and look at the URL. Most WordPress sites follow:
- `site.com/category/business/page/2/`  → pattern is `site.com/category/business/page/{}/`
- `site.com/news/business?page=2`       → pattern is `site.com/news/business?page={}`

The `{}` is where Python will insert the page number automatically.

---

## Understanding Website Configuration in BeautifulSoup

When scraping websites, it is common to create a **configuration dictionary** that stores information needed to locate and extract content.

Instead of hardcoding selectors inside the scraping logic, we store them separately so the scraper becomes easier to maintain and reuse.

The examples below use **Daily News** as a sample website, but these ideas apply to most websites.

### 1. `name`

A simple label used to identify the website source.

```python
"Daily News"
```

Useful when scraping from multiple sources because it lets you track where collected data came from.

---

### 2. `base_url`

Stores the page URL structure.

```python
"https://dailynews.co.tz/category/business/page/{}/"
```

The `{}` acts as a placeholder for page numbers.

Examples:

* `page/1`
* `page/2`
* `page/3`

This allows the scraper to automatically move from one page to another while collecting data.

---

### 3. `article_container`

CSS selector:

```python
#masonry-grid
```

`#` means select an element by **ID**.

Looks for:

```html
<div id="masonry-grid">
```

The scraper first enters this section of the page so it focuses only on the area containing articles instead of searching the entire website.

---

### 4. `article_link_sel`

CSS selector:

```python
h2.entry-title a
```

Breakdown:

* `h2` → select `<h2>` tags
* `.entry-title` → only those with class `entry-title`
* `a` → links inside them

The scraper uses this path to locate clickable article headlines and collect their titles and links.

---

### 5. `fallback_link_sel`

Backup selector:

```python
h2 a, h3 a
```

The comma means **OR**.

Looks for links inside:

* `<h2>`
* `<h3>`

This acts as a backup option so the scraper can still find article links if the main page structure changes.

---

### 6. `date_sel`

CSS selector:

```python
span.date
```

Targets:

```html
<span class="date">
```

The scraper looks for date elements and extracts the text inside them so publication dates can be saved.

---

### 7. `category_filter`

Example:

```python
None
```

This means no filtering is applied.

Since the example page already contains business articles, the scraper collects everything without needing extra filtering rules.

---

### 8. `base_domain`

Example:

```python
''
```

Used when websites provide **relative URLs**.

Example:

```html
<a href="/article/123">
```

This helps convert incomplete links into full website URLs so they can be opened correctly.

---

### Why This Matters

Think of this dictionary as a map for your scraper. It tells the program where to look and what to collect from a website, making the code easier to update and reuse.


In [4]:
# SITE CONFIGURATIONS — edit here to add new sites

DAILY_NEWS = {
    'name'             : 'Daily News',
    'base_url'         : 'https://dailynews.co.tz/category/business/page/{}/',
    'article_container': '#masonry-grid',
    'article_link_sel' : 'h2.entry-title a',
    'fallback_link_sel': 'h2 a, h3 a',
    'date_sel'         : 'span.date',          # <span class="date meta-item tie-icon">
    'category_filter'  : [],                 # no filtering needed, all posts are business
    'base_domain'      : '',                   # links are already absolute
}

THE_CITIZEN = {
    'name'             : 'The Citizen',
    'base_url'         : 'https://www.thecitizen.co.tz/service/search/tanzania/2718734?pageNum={}&query=business&sortByDate=true',
    'article_container': None,
    'article_link_sel' : 'a.teaser-image-left',
    'fallback_link_sel': 'h3 a',
    'date_sel'         : 'span.date',          # same class, different content
    'category_filter'  : ['business','national'],           # skip OpEd and etc.
    'base_domain'      : 'https://www.thecitizen.co.tz',  # links are relative
}

# ── To add a new site, copy the template below and fill in the values ──
# NEW_SITE = {
#     'name'              : 'Site Name',
#     'base_url'          : 'https://example.com/category/page/{}/',
#     'article_container' : '#container-id or .container-class',
#     'article_link_sel'  : 'h2.entry-title a',
#     'fallback_link_sel' : 'h2 a, h3 a',
# }

print("✓ Site configs loaded")
print(f"  → {DAILY_NEWS['name']}: {DAILY_NEWS['base_url']}")
print(f"  → {THE_CITIZEN['name']}: {THE_CITIZEN['base_url']}")


✓ Site configs loaded
  → Daily News: https://dailynews.co.tz/category/business/page/{}/
  → The Citizen: https://www.thecitizen.co.tz/service/search/tanzania/2718734?pageNum={}&query=business&sortByDate=true


---
## Section 2: Core Scraping Functions

These functions do all the heavy lifting. You should **never need to edit these**
unless you want to change the scraping behaviour globally.

| Function | What it does |
|----------|-------------|
| `scrape_article(url, source)` | Downloads one article page and extracts headline + date |
| `get_article_urls(page_html, config)` | Extracts all article links from one listing page |
| `scrape_site(config, start_page, end_page, output_file)` | Main loop — scrapes all pages and saves to CSV |
| `append_to_csv(records, filename)` | Saves records without overwriting existing data |

> 🔧 **Functions will be defined here in the next step.**


In [2]:
import requests
from bs4 import BeautifulSoup
import csv
import os
import time
from datetime import datetime


# ══════════════════════════════════════════════════════
# FUNCTION 1: Parse date string to YYYY-MM-DD
# ══════════════════════════════════════════════════════

def parse_date(raw_date):
    """
    Converts a human-readable date string to YYYY-MM-DD format.

    Handles the format both Daily News and The Citizen use: "Aug 21, 2023"
    If parsing fails (e.g. relative dates like "Yesterday", "6 days ago"),
    the raw string is returned as-is so no data is lost.

    Args:
        raw_date : raw date string from the listing page

    Returns:
        "YYYY-MM-DD" string if parsing succeeds, otherwise the original string.
    """
    if not raw_date:
        return 'unknown'

    # Clean off anything after a dash e.g. "Aug 21, 2023 - 3 min read"
    cleaned = raw_date.split('-')[0].strip()

    try:
        return datetime.strptime(cleaned, "%B %d, %Y").strftime("%Y-%m-%d")  # "March 14, 2026"
    except ValueError:
        pass

    try:
        return datetime.strptime(cleaned, "%b %d, %Y").strftime("%Y-%m-%d")  # "Aug 21, 2023"
    except ValueError:
        pass

    # Couldn't parse — return raw string (handles relative dates gracefully)
    return cleaned


# ══════════════════════════════════════════════════════
# FUNCTION 2: Extract article links, headlines and dates
#             from one listing page
# ══════════════════════════════════════════════════════

def get_article_urls(page_html, config):
    """
    Extracts article headlines, links, and dates from one listing page.
    No article pages are opened — everything comes from the listing page only.

    Strategy:
    - Tries article_container first to narrow search scope
    - Falls back to full page search if container not found
    - Filters by category if config['category_filter'] is set (The Citizen)
    - Grabs headline from the link text directly
    - Grabs date from config['date_sel'] near each card

    Args:
        page_html : raw HTML string of the listing page
        config    : site config dict (DAILY_NEWS or THE_CITIZEN)

    Returns:
        List of dicts with keys: 'headline', 'url', 'date'
    """
    soup = BeautifulSoup(page_html, 'html.parser')
    results = []

    # ── Step 1: Find search scope ──
    container = None
    if config['article_container']:
        container = soup.select_one(config['article_container'])

    if container:
        search_scope = container
        selector = config['article_link_sel']
    else:
        if config['article_container']:
            # Was defined but not found on this page
            print(f"    ⚠ Container '{config['article_container']}' not found — using fallback")
            selector = config['fallback_link_sel']
        else:
            # Intentionally None (The Citizen) — use article_link_sel directly
            selector = config['article_link_sel']
        search_scope = soup

    # ── Step 2: Find all article cards ──
    cards = search_scope.select(selector)

    if not cards:
        print(f"    ⚠ No articles found with selector '{selector}'")
        return results

    # ── Step 3: Loop through each card ──
    for card in cards:

        # 3a: Category filtering (The Citizen only)
        if config['category_filter']:
            category_tag = card.select_one('span.article-topic')
            if not category_tag:
                continue
            if category_tag.get_text(strip=True).lower() not in config['category_filter']:
                continue

        # 3b: Get URL
        if card.name == 'a':
            relative_url = card.get('href', '').strip()
        else:
            inner_link = card.find('a')
            relative_url = inner_link.get('href', '').strip() if inner_link else ''

        if not relative_url:
            continue

        full_url = config['base_domain'] + relative_url

        # 3c: Get headline from link text — no article visit needed
        headline = card.get_text(strip=True)

        # For Daily News the <a> tag wraps only the headline text
        # For The Citizen the <a> wraps the whole card so we target the h3
        if config['name'] == 'The Citizen':
            h3 = card.select_one('h3.title-small')
            headline = h3.get_text(strip=True) if h3 else headline

        if not headline:
            continue

        # 3d: Get date from nearest parent card wrapper
        date = 'unknown'
        parent = card.find_parent(['article', 'li', 'div'])
        if parent:
            date_tag = parent.select_one(config['date_sel'])
            if date_tag:
                raw_date = date_tag.get_text(strip=True)
                date = parse_date(raw_date)

        results.append({
            'headline' : headline,
            'url'      : full_url,
            'date'     : date,
        })

    return results


# ══════════════════════════════════════════════════════
# FUNCTION 3: Save records to CSV
# ══════════════════════════════════════════════════════

def append_to_csv(records, filename):
    """
    Appends a list of article records to a CSV file.

    Creates the file with a header row on first write.
    On subsequent writes, appends without repeating the header.
    Safe to call after every page or checkpoint — no data is ever overwritten.

    Args:
        records  : list of dicts with keys: date, headline, url, source, scraped_on
        filename : path to the output CSV file
    """
    if not records:
        return

    file_exists = os.path.isfile(filename)

    with open(filename, mode='a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['date', 'headline', 'url', 'source', 'scraped_on'])
        if not file_exists:
            writer.writeheader()
        writer.writerows(records)


# ══════════════════════════════════════════════════════
# FUNCTION 4: Main scraping loop
# ══════════════════════════════════════════════════════

def scrape_site(config, start_page, end_page, output_file):
    """
    Main loop. Scrapes every listing page from start_page to end_page,
    extracts headlines, dates, and URLs entirely from listing pages,
    and saves to CSV with checkpoints every 20 pages.

    Resumable: increase start_page to last successful page after a crash.
    Checkpoint saves happen every 20 pages so you never lose more than
    20 pages of work.

    Args:
        config      : site config dict (DAILY_NEWS or THE_CITIZEN)
        start_page  : first page number to scrape
        end_page    : last page number to scrape (inclusive)
        output_file : filename for the output CSV
    """
    print(f"\n{'='*55}")
    print(f"  Starting scrape : {config['name']}")
    print(f"  Pages           : {start_page} → {end_page}")
    print(f"  Output          : {output_file}")
    print(f"{'='*55}\n")

    # Today's date stamped on every record so you know when this run happened
    scraped_on = datetime.today().strftime("%Y-%m-%d")

    total_saved  = 0
    batch_records = []   # accumulates records until checkpoint

    for page_num in range(start_page, end_page + 1):

        page_url = config['base_url'].format(page_num)
        print(f"[Page {page_num}] Fetching: {page_url}")

        # ── Fetch listing page ──
        try:
            response = requests.get(page_url, timeout=10, headers={
                'User-Agent': 'Mozilla/5.0 (compatible; research-scraper/1.0)'
            })

            # 404 means we've gone past the last page — stop cleanly
            if response.status_code == 404:
                print(f"  404 reached at page {page_num} — end of archive, stopping\n")
                break

            response.raise_for_status()

        except requests.RequestException as e:
            print(f"  ⚠ WARNING: Could not load page {page_num} — skipping")
            print(f"    Reason: {e}\n")
            continue

        # ── Extract articles from listing page ──
        articles = get_article_urls(response.text, config)

        if not articles:
            print(f"  ⚠ WARNING: No articles found on page {page_num} — skipping\n")
            continue

        # ── Build records and add metadata ──
        for article in articles:
            batch_records.append({
                'date'      : article['date'],
                'headline'  : article['headline'],
                'url'       : article['url'],
                'source'    : config['name'],
                'scraped_on': scraped_on,
            })

        total_saved += len(articles)
        print(f"  ✓ {len(articles)} articles collected (total so far: {total_saved})")

        # ── Checkpoint save every 20 pages ──
        if page_num % 20 == 0:
            append_to_csv(batch_records, output_file)
            batch_records = []   # clear batch after saving
            print(f"\tCheckpoint saved at page {page_num} — {total_saved} records in {output_file}")

        time.sleep(1)   # be polite between listing page requests

    # ── Save any remaining records that didn't hit a checkpoint ──
    if batch_records:
        append_to_csv(batch_records, output_file)
        print(f"\n\tFinal save: {len(batch_records)} remaining records written\n")

    print(f"\n{'='*55}")
    print(f"  ✓ Scrape complete!")
    print(f"  Total headlines saved : {total_saved}")
    print(f"  File                  : {output_file}")
    print(f"{'='*55}\n")

---
## Section 3: Run the Scraper

This is where you actually collect data. Three steps:

1. Choose your site config from Section 3
2. Set your page range (check the site manually to find which pages cover 2022–2024)
3. Set your output filename and run

To **resume** from where you left off after a crash, just change `start_page`
to the last page that successfully saved. The append function ensures no data is lost
and no duplicate headers are written.


In [5]:
# ══════════════════════════════════════════════════════
# RUN CONFIGURATION — change these values as needed
# ══════════════════════════════════════════════════════


# citizen pages btn 424 to 841
# daily news btn 287 to 683

scrape_site(THE_CITIZEN,425,426,"test.csv")


  Starting scrape : The Citizen
  Pages           : 425 → 426
  Output          : test.csv

[Page 425] Fetching: https://www.thecitizen.co.tz/service/search/tanzania/2718734?pageNum=425&query=business&sortByDate=true
  ✓ 5 articles collected (total so far: 5)
[Page 426] Fetching: https://www.thecitizen.co.tz/service/search/tanzania/2718734?pageNum=426&query=business&sortByDate=true
  ✓ 6 articles collected (total so far: 11)

	Final save: 11 remaining records written


  ✓ Scrape complete!
  Total headlines saved : 11
  File                  : test.csv



In [8]:
# citizen pages btn 424 to 841
# daily news btn 287 to 683

scrape_site(DAILY_NEWS,287,683,"final_news_scrape.csv")


  Starting scrape : Daily News
  Pages           : 287 → 683
  Output          : final_news_scrape.csv

[Page 287] Fetching: https://dailynews.co.tz/category/business/page/287/
  ✓ 10 articles collected (total so far: 10)
[Page 288] Fetching: https://dailynews.co.tz/category/business/page/288/
  ✓ 10 articles collected (total so far: 20)
[Page 289] Fetching: https://dailynews.co.tz/category/business/page/289/
  ✓ 10 articles collected (total so far: 30)
[Page 290] Fetching: https://dailynews.co.tz/category/business/page/290/
  ✓ 10 articles collected (total so far: 40)
[Page 291] Fetching: https://dailynews.co.tz/category/business/page/291/
  ✓ 10 articles collected (total so far: 50)
[Page 292] Fetching: https://dailynews.co.tz/category/business/page/292/
  ✓ 10 articles collected (total so far: 60)
[Page 293] Fetching: https://dailynews.co.tz/category/business/page/293/
  ✓ 10 articles collected (total so far: 70)
[Page 294] Fetching: https://dailynews.co.tz/category/business/page/2

In [ ]:


SITE        = DAILY_NEWS          # ← swap to THE_CITIZEN or any other config
START_PAGE  = 287                 # ← first page of your target date range
END_PAGE    = 683                 # ← last page of your target date range
OUTPUT_FILE = 'dailynews_headlines_2022_2024.csv'

# ── To resume after a crash, update START_PAGE to where it stopped ──
# START_PAGE = 450   # example: resume from page 450

print(f"Ready to scrape: {SITE['name']}")
print(f"Pages          : {START_PAGE} → {END_PAGE}")
print(f"Output file    : {OUTPUT_FILE}")


In [ ]:
# ── Start scraping ──
# scrape_site(SITE, START_PAGE, END_PAGE, OUTPUT_FILE)

print("⚠️  Uncomment the line above once functions in Section 4 are defined")


---
## Section 6 — Results Preview

Once scraping is complete, run the cells below to inspect and clean your dataset.


In [ ]:
# ── Load and preview the collected dataset ──
df = pd.read_csv(OUTPUT_FILE)

print(f"Total rows collected : {len(df)}")
print(f"Columns              : {list(df.columns)}")
print(f"\nDate range           : {df['date'].min()} → {df['date'].max()}")


In [ ]:
# ── Year distribution ──
df['year'] = df['date'].str[:4]
print("Articles per year:")
print(df['year'].value_counts().sort_index())


In [ ]:
# ── Remove duplicates ──
before = len(df)
df = df.drop_duplicates(subset='url')
after  = len(df)
print(f"Duplicates removed: {before - after}")
print(f"Clean dataset size: {after} articles")

df.to_csv(OUTPUT_FILE, index=False)
print(f"\n✓ Saved clean dataset to {OUTPUT_FILE}")


In [ ]:
# ── Sample headlines ──
df.sample(10)[['date', 'headline', 'source']]


---
## Section 7 — Next Steps

With your headlines dataset ready, the next notebooks in this project will cover:

1. **Sentiment Analysis** — label each headline as Positive / Negative / Neutral
   using a pre-trained model (e.g. VADER or a fine-tuned transformer)

2. **Financial Data Collection** — download TZS/USD exchange rates from the
   Bank of Tanzania (`bot.go.tz`) and CPI/inflation data from NBS Tanzania (`nbs.go.tz`)

3. **Correlation Analysis** — merge sentiment scores with financial indicators
   and analyse whether news tone correlates with currency or inflation movements

4. **Visualisation & Reporting** — charts, heatmaps, and summary statistics
   for the final project report

---
*Tanzania Financial News Scraper — DS & AI Year I Project*
